# Feature names through selection on Titanic

Replay the feature-name example from the scikit-learn 1.1 release highlights on the original OpenML Titanic dataset. Median imputation and scaling handle age and fare. Dense one-hot encoding handles embarkation and passenger class. The ANOVA selector retains seven columns whose names label the fitted logistic regression coefficients.

Moxie uses a named `ColumnTable` and one encoder per categorical domain, so passenger class stays integer-valued and missing embarkation stays distinct from observed strings. The fitted preprocessing and selection stages form a typed transformer prefix. Querying its names corresponds to scikit-learn's `model[:-1].get_feature_names_out()`. The native pipeline shares Float64 binary labels across its prefix and classifier.

Adapted from the scikit-learn developers' [1.1 release highlights](https://github.com/scikit-learn/scikit-learn/blob/4358542a1bd39af2e7e1cbace8b9fed067473c5b/examples/release_highlights/plot_release_highlights_1_1_0.py) (BSD-3-Clause).

In [1]:
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.feature_selection import SelectKBest
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/_support/moxie_nb.py").exists())
oracle = ROOT / "families/ensemble/builds/oracle-env"
receipt = ROOT / "families/ensemble/docs/private/forest_engine/oracle-build.json"
if oracle not in Path(sklearn.__file__).resolve().parents:
    raise RuntimeError("Use the pinned oracle described in docs/feature_selection/classifier-selection.md")
assert json.loads(receipt.read_text())["revision"] == "4358542a1bd39af2e7e1cbace8b9fed067473c5b"

sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("release_highlights/titanic_feature_names")

X, y = fetch_openml(
    "titanic", version=1, as_frame=True, return_X_y=True, parser="pandas",
    data_home=ROOT / "builds/notebooks/openml",
)
numeric_features = ["age", "fare"]
preprocessor = ColumnTransformer(
    [
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["embarked", "pclass"]),
    ],
    verbose_feature_names_out=False,
)
reference = make_pipeline(preprocessor, SelectKBest(k=7), LogisticRegression()).fit(X, y)
embarked = X["embarked"].astype(object)
nb.share(
    numeric=X[numeric_features].to_numpy(),
    embarked=embarked.fillna("").to_numpy(),
    embarked_missing=embarked.isna().to_numpy(),
    pclass=X["pclass"].to_numpy(dtype=np.int64),
    y=y.to_numpy(dtype=np.float64),
)

In [2]:
%%mojo titanic_feature_names
from moxie._categorical import categorical_column
from moxie.compose import ColumnEntry, ColumnTable, ColumnTransformer
from moxie.feature_selection import BuiltinScorer, SelectKBest, SelectionParameter
from moxie.impute import SimpleImputer
from moxie.linear_model import LogisticRegression
from moxie.pipeline import make_pipeline
from moxie.preprocessing import OneHotEncoder, StandardScaler
from moxie.preprocessing._vocabulary import NumericKind, StringNaNKind

from moxie_nb_io import load, load_strings, save, save_strings


def main() raises:
    var numeric_names: List[String] = ["age", "fare"]
    var embarked_names: List[String] = ["embarked"]
    var class_names: List[String] = ["pclass"]
    var table = ColumnTable(load("numeric"), numeric_names)
    var embarked = load_strings("embarked")
    var embarked_missing = load[DType.bool]("embarked_missing")
    var embarked_cells = List[Optional[String]](capacity=len(embarked))
    for i in range(len(embarked)):
        embarked_cells.append(None if embarked_missing.item(i) else Optional[String](embarked[i]))
    table.add_categorical("embarked", categorical_column(embarked_cells))
    table.add_numeric("pclass", load[DType.int64]("pclass"))

    var preprocessor = ColumnTransformer(
        ColumnEntry(
            "num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_names^,
        ),
        ColumnEntry(
            "embarked", OneHotEncoder[StringNaNKind](handle_unknown="ignore", sparse_output=False),
            embarked_names^,
        ),
        ColumnEntry(
            "pclass", OneHotEncoder[NumericKind[DType.int64]](handle_unknown="ignore", sparse_output=False),
            class_names^,
        ),
        verbose_feature_names_out=False,
    )
    var prefix = make_pipeline(
        preprocessor^.as_table_transformer(),
        SelectKBest(BuiltinScorer("f_classif"), k=SelectionParameter(7)),
    )
    var model = make_pipeline(prefix^, LogisticRegression[label_dtype=DType.float64]())
    _ = model.fit(table, load("y"))

    save_strings("preprocessed_names", model.transformer.transformer.get_feature_names_out())
    save_strings("selected_names", model.transformer.get_feature_names_out())
    save("selected", model.transformer.transform(table))
    save("coefficients", model.estimator.coef_)
    save("predictions", model.predict(table))


In [3]:
native = nb.results["titanic_feature_names"]
prefix = reference[:-1]
coefficients = reference[-1].coef_
nb.compare("Preprocessor names", preprocessor.get_feature_names_out(), native["preprocessed_names"], exact=True)
nb.compare("Selected names in coefficient order", prefix.get_feature_names_out(), native["selected_names"], exact=True)
nb.compare("Selected training columns", prefix.transform(X), native["selected"],
           rtol=1e-12, atol=1e-12, reason="Float64 scaling reductions can round differently")
nb.compare("Logistic coefficients", coefficients, native["coefficients"],
           rtol=0, atol=1e-6, reason="Independent L-BFGS line search and convergence at default tolerance")
nb.compare("Training predictions", reference.predict(X).astype(np.float64), native["predictions"], exact=True)
nb.report()
nb.assert_parity()

pd.DataFrame({
    "feature": native["selected_names"],
    "scikit-learn": coefficients.ravel(),
    "Moxie": native["coefficients"].ravel(),
})

check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
Preprocessor names,PASS,"(9,)","(9,)",,,exact,0 mismatches,
Selected names in coefficient order,PASS,"(7,)","(7,)",,,exact,0 mismatches,
Selected training columns,PASS,"(1309, 7)","(1309, 7)",7.11e-15,7.13e-13,"rtol=1e-12, atol=1e-12",,Float64 scaling reductions can round differently
Logistic coefficients,PASS,"(1, 7)","(1, 7)",1.41e-07,1.59e-05,"rtol=0, atol=1e-06",,Independent L-BFGS line search and convergence at default tolerance
Training predictions,PASS,"(1309,)","(1309,)",,,exact,0 mismatches,


,feature,scikit-learn,Moxie
0,fare,0.249689,0.249689
1,embarked_C,0.008889,0.008890
2,embarked_S,-0.594846,-0.594846
3,embarked_nan,0.447542,0.447542
4,pclass_1,0.490438,0.490438
5,pclass_2,0.201301,0.201301
6,pclass_3,-0.660623,-0.660623
